# 모델·손실·학습을 unit 단위로 확인
실제 DINOv3 블록을 작은 규격으로 구성하여 CPU에서 실행합니다. 각 셀을 Shift+Enter로 실행합니다. 실제 ViT-S 초기화는 02 노트북에서 수행합니다. 패키지는 README의 설치 셀로 설치합니다.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd()
if not (ROOT / 'pyproject.toml').exists():
    ROOT = ROOT.parent
assert (ROOT / 'pyproject.toml').exists(), 'Open this notebook from the repository'
sys.path.insert(0, str(ROOT / 'src'))
import torch
torch.set_num_threads(2)
torch.manual_seed(20261112)
print('Repository', ROOT.name, '| PyTorch', torch.__version__)

## 1. 실제 DINOv3 인코더와 CPDINO 헤드 연결

In [ ]:
from dinov3.models.vision_transformer import DinoVisionTransformer
from cpdino_lab.models import adapt_encoder, build_model
encoder = DinoVisionTransformer(img_size=32, patch_size=16, embed_dim=48, depth=2,
    num_heads=3, n_storage_tokens=4, layerscale_init=1e-5, pos_embed_rope_dtype='fp32')
encoder.init_weights()
student = adapt_encoder(encoder, 'unit-demo', rdrop=0)
images = torch.rand(1, 3, 32, 48)
labels = torch.zeros(1, 3, 32, 48)
prediction, _ = student(images)
assert prediction.shape == (1, 3, 32, 48)
print('Output', tuple(prediction.shape))

## 2. 공식 크기 S/B/L의 구조 비교
메모리 할당을 생략하는 meta device에서 실제 팩토리 구조를 조회합니다.

In [ ]:
for name in ['vitl','vitb','vits']:
    with torch.device('meta'):
        model = build_model(name)
    print(name, 'blocks', len(model.encoder.blocks), 'readout', tuple(model.out.weight.shape),
          'encoder parameters', sum(p.numel() for p in model.encoder.parameters()))
del model

## 3. 라벨 손실과 교사 출력 손실

In [ ]:
import copy
from cpdino_lab.losses import segmentation_loss
teacher = copy.deepcopy(student).eval()
with torch.no_grad():
    teacher.out.bias.add_(0.5)
    target = teacher(images)[0]
task = segmentation_loss(prediction, labels)
distilled = segmentation_loss(prediction, labels, target, teacher_weight=1)
assert distilled > task
print({'supervised_loss':float(task.detach()), 'with_distillation':float(distilled.detach())})

## 4. 한 번의 optimizer step
학생 헤드가 변하고 교사는 gradient를 받지 않는지 확인합니다.

In [ ]:
from cpdino_lab.engine import train_step, evaluate
optimizer = torch.optim.AdamW(student.parameters(), lr=1e-3)
before = student.out.weight.detach().clone()
loss = train_step(student, optimizer, images, labels, teacher)
assert not torch.equal(before, student.out.weight)
assert all(p.grad is None for p in teacher.parameters())
print('Training loss', loss, '| validation', evaluate(student, [(images, labels)]))

## 5. 체크포인트 저장과 재개

In [ ]:
from cpdino_lab.checkpoints import save_checkpoint, restore_checkpoint
checkpoint = ROOT / 'runs/unit-demo/last.pt'
identity = {'experiment':'unit-demo', 'seed':20261112}
save_checkpoint(checkpoint, student, optimizer, epoch=1, identity=identity)
epoch = restore_checkpoint(checkpoint, student, optimizer, identity)
assert epoch == 1
print('Restored epoch', epoch)

## 6. 모듈별 자동 검증

In [ ]:
import pytest
result = pytest.main([str(ROOT / 'tests'), '-q'])
assert result == pytest.ExitCode.OK